In [1]:
!pip install pyspark

In [2]:
# -----------------------------------------
# Step 0: Install PySpark
# -----------------------------------------
!pip install pyspark --quiet

# -----------------------------------------
# Step 1: Imports
# -----------------------------------------
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from sklearn.datasets import fetch_california_housing
import pandas as pd

# Start Spark
spark = SparkSession.builder.appName("PySpark_CaliforniaHousing").getOrCreate()

# -----------------------------------------
# Step 2: Load California Housing dataset
# -----------------------------------------
california = fetch_california_housing()
df = pd.DataFrame(california.data, columns=california.feature_names)
df['target'] = california.target  # Median house value

# Convert to PySpark DataFrame
df_reg = spark.createDataFrame(df)
df_reg.show(5)
df_reg.printSchema()

# -----------------------------------------
# Step 3: Prepare features
# -----------------------------------------
feature_cols = california.feature_names
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

train, test = df_reg.randomSplit([0.8, 0.2], seed=42)
train_assembled = assembler.transform(train)
test_assembled = assembler.transform(test)

# -----------------------------------------
# Step 4: Random Forest Regressor
# -----------------------------------------
rf = RandomForestRegressor(labelCol="target", featuresCol="features", numTrees=50)
rf_model = rf.fit(train_assembled)
rf_preds = rf_model.transform(test_assembled)

# -----------------------------------------
# Step 5: Gradient-Boosted Trees Regressor (XGBoost-style)
# -----------------------------------------
gbt = GBTRegressor(labelCol="target", featuresCol="features", maxIter=100)
gbt_model = gbt.fit(train_assembled)
gbt_preds = gbt_model.transform(test_assembled)

# -----------------------------------------
# Step 6: Evaluate models
# -----------------------------------------
reg_eval = RegressionEvaluator(labelCol="target", predictionCol="prediction", metricName="rmse")
print("Random Forest RMSE:", reg_eval.evaluate(rf_preds))
print("GBT RMSE:", reg_eval.evaluate(gbt_preds))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/25 19:20:17 WARN Utils: Your hostname, alexs-Mac-mini.local, resolves to a loopback address: 127.0.0.1; using 100.98.208.105 instead (on interface utun4)
26/03/25 19:20:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/25 19:20:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Traceback (most recent call last):                                  (0 + 1) / 1]
  File "/Volumes/T7 Shield/develop/bigdata-project/.venv/lib/python3.14/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
  File "/Volumes/T7 Shield/develop/bigd

+------+--------+------------------+------------------+----------+------------------+--------+---------+------+
|MedInc|HouseAge|          AveRooms|         AveBedrms|Population|          AveOccup|Latitude|Longitude|target|
+------+--------+------------------+------------------+----------+------------------+--------+---------+------+
|8.3252|    41.0| 6.984126984126984|1.0238095238095237|     322.0|2.5555555555555554|   37.88|  -122.23| 4.526|
|8.3014|    21.0| 6.238137082601054|0.9718804920913884|    2401.0| 2.109841827768014|   37.86|  -122.22| 3.585|
|7.2574|    52.0| 8.288135593220339| 1.073446327683616|     496.0|2.8022598870056497|   37.85|  -122.24| 3.521|
|5.6431|    52.0|5.8173515981735155|1.0730593607305936|     558.0| 2.547945205479452|   37.85|  -122.25| 3.413|
|3.8462|    52.0| 6.281853281853282|1.0810810810810811|     565.0|2.1814671814671813|   37.85|  -122.25| 3.422|
+------+--------+------------------+------------------+----------+------------------+--------+---------+

26/03/25 19:20:48 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
